# Random Forest classification development

This notebook uses the **same complete hourly dataset and reproducible station-based split as `sets.ipynb`**: 420,768 source rows, 336,612 training rows, and 84,156 test rows. It trains separate Random Forest classifiers for the overall daily China AQI category and the daily category for each pollutant, using the breakpoint and averaging rules documented in `sets.ipynb`.

The daily label lookup has about 17,000 unique station-day rows; that is **not** the model dataset size. Each daily label is attached to its matching original hourly rows. Models train on the 336,612 hourly training rows and evaluate on the 84,156 hourly test rows before rows without a valid daily label are excluded. Pollutant concentrations, pollutant averages, IAQIs, AQI, and label-derived fields are excluded from predictors to avoid direct target leakage. Predictors are hourly meteorology, station, and calendar/time context. Missing numeric predictors are median-imputed; categorical predictors are filled and one-hot encoded. Random Forest does not require feature scaling.

**Evaluation caveat:** the requested split is by hourly row, while the targets are daily. Hours from one station-day can therefore occur in both train and test. The reported test metrics evaluate the requested hourly-row split, not an independent station-day holdout.


## 1. Load and confirm the modeling inputs

Load the same `all_regions` MySQL table as `sets.ipynb`. This notebook checks the source and split sizes against the counts you provided before proceeding.


In [ ]:
import os
import re
import numpy as np
import pandas as pd

from IPython.display import display
from sqlalchemy import create_engine, text

try:
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, balanced_accuracy_score
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder
except ImportError as error:
    raise ImportError(
        "This modeling notebook requires scikit-learn. Install it in this notebook's "
        "Python environment with `%pip install scikit-learn`, then restart the kernel."
    ) from error

RANDOM_STATE = 42
TEST_FRACTION = 0.20
SOURCE_TABLE = "all_regions"
EXPECTED_SOURCE_ROWS = 420_768
EXPECTED_TRAIN_ROWS = 336_612
EXPECTED_TEST_ROWS = 84_156
DB_URL = os.environ.get("AIR_QUALITY_DB_URL", "mysql+pymysql://root:@localhost/air_quality")

if not re.fullmatch(r"[A-Za-z0-9_]+", SOURCE_TABLE):
    raise ValueError(f"Invalid MySQL table name: {SOURCE_TABLE!r}")
engine = create_engine(DB_URL, pool_pre_ping=True)
try:
    with engine.connect() as connection:
        source_df = pd.read_sql_query(text(f"SELECT * FROM {SOURCE_TABLE}"), connection)
except Exception as error:
    raise RuntimeError(
        f"Could not load {SOURCE_TABLE!r} from the air_quality database. Check MySQL, "
        "AIR_QUALITY_DB_URL, the table name, SQLAlchemy, and PyMySQL. "
        f"Original database error: {error}"
    ) from error

print(f"Loaded source rows: {len(source_df):,}")
if len(source_df) != EXPECTED_SOURCE_ROWS:
    raise ValueError(
        f"Expected {EXPECTED_SOURCE_ROWS:,} rows in air_quality.{SOURCE_TABLE}, "
        f"but found {len(source_df):,}. The source must match the dataset used for "
        "the requested 336,612/84,156 split."
    )
display(source_df.head())
display(pd.DataFrame({
    "dtype": source_df.dtypes.astype(str),
    "missing_rows": source_df.isna().sum(),
    "missing_percent": (source_df.isna().mean() * 100).round(2),
}))

POLLUTANT_BREAKPOINTS = {
    "PM2.5": [0, 30, 60, 115, 150, 250, 350, 500],
    "PM10": [0, 50, 120, 250, 350, 420, 500, 600],
    "SO2": [0, 50, 150, 475, 800, 1600, 2100, 2620],
    "NO2": [0, 40, 80, 180, 280, 565, 750, 940],
    "CO": [0, 2, 4, 14, 24, 36, 48, 60],
    "O3": [0, 100, 160, 215, 265, 800],
}
AQI_BREAKPOINTS = [0, 50, 100, 150, 200, 300, 400, 500]
O3_1H_BREAKPOINTS = [0, 160, 200, 300, 400, 800, 1000, 1200]
CATEGORY_NAMES = {
    1: "Excellent", 2: "Good", 3: "Lightly polluted",
    4: "Moderately polluted", 5: "Heavily polluted", 6: "Severely polluted",
}
POLLUTANTS = list(POLLUTANT_BREAKPOINTS)
required_columns = [*POLLUTANTS, "station", "year", "month", "day", "hour"]
missing_columns = [column for column in required_columns if column not in source_df.columns]
if missing_columns:
    raise KeyError(f"The source table is missing required columns: {missing_columns}")


## 2. Calculate daily China AQI and pollutant category targets

This applies the same rules as `sets.ipynb`: 24-hour pollutants require at least 20 valid hourly readings; ozone uses the maximum qualifying trailing 8-hour mean, with the 1-hour ozone breakpoint rule above 800 µg/m³. The daily label table is a station-day lookup only (about 17,000 rows). Each label is joined back to matching hourly rows, so the classifiers use the 336,612/84,156 hourly split—not a 17,000-row model dataset. Invalid or unavailable daily targets remain missing and are excluded from that target’s model fit.


In [ ]:
model_source = source_df.copy()
time_parts = model_source[["year", "month", "day", "hour"]].apply(pd.to_numeric, errors="coerce")
timestamps = pd.to_datetime({
    "year": time_parts["year"], "month": time_parts["month"],
    "day": time_parts["day"], "hour": time_parts["hour"],
}, errors="coerce")
model_source["_timestamp"] = timestamps
model_source["_date"] = timestamps.dt.normalize()
model_source["_station_key"] = model_source["station"].astype("string").str.strip().str.casefold()
model_source = model_source.loc[
    model_source["_timestamp"].notna() & model_source["_station_key"].notna()
].copy()
if model_source.duplicated(["_station_key", "_timestamp"]).any():
    raise ValueError("Duplicate station-hour rows were found; resolve them before making daily labels.")
for pollutant in POLLUTANTS:
    values = pd.to_numeric(model_source[pollutant], errors="coerce")
    model_source[pollutant] = values.mask(values.lt(0))

station_day_keys = ["_station_key", "_date"]
daily_group = model_source.groupby(station_day_keys, sort=False)
daily_targets = daily_group[["station"]].first()
daily_targets.index.names = station_day_keys
for pollutant in ["PM2.5", "PM10", "SO2", "NO2", "CO"]:
    valid_counts = daily_group[pollutant].count()
    daily_mean = daily_group[pollutant].mean()
    daily_targets[f"{pollutant}_24h_avg"] = daily_mean.where(valid_counts.ge(20))

o3_windows = []
for _, station_rows in model_source.groupby("_station_key", sort=False):
    ordered = station_rows.sort_values("_timestamp")
    ozone = pd.Series(ordered["O3"].to_numpy(), index=pd.DatetimeIndex(ordered["_timestamp"]))
    rolling_8h = ozone.rolling("8h", closed="right", min_periods=6).mean()
    o3_windows.append(pd.DataFrame({
        "_station_key": ordered["_station_key"].to_numpy(),
        "_date": ordered["_date"].to_numpy(),
        "_hour": ordered["_timestamp"].dt.hour.to_numpy(),
        "O3_8h_avg": rolling_8h.to_numpy(),
    }))
o3_windows = pd.concat(o3_windows, ignore_index=True)
o3_windows = o3_windows.loc[o3_windows["_hour"].between(8, 23)]
o3_by_day = o3_windows.groupby(station_day_keys, sort=False)["O3_8h_avg"]
o3_max_8h = o3_by_day.max()
o3_valid_windows = o3_by_day.count()
daily_targets["O3_8h_avg"] = o3_max_8h.where(o3_valid_windows.ge(14))
daily_targets["O3_1h_max"] = daily_group["O3"].max()

def interpolate_iaqi(concentration, concentration_breakpoints):
    result = pd.Series(np.nan, index=concentration.index, dtype="float64")
    for i in range(len(concentration_breakpoints) - 1):
        bp_low, bp_high = concentration_breakpoints[i:i + 2]
        iaqi_low, iaqi_high = AQI_BREAKPOINTS[i:i + 2]
        in_range = concentration.gt(bp_low) & concentration.le(bp_high)
        result.loc[in_range] = (
            (iaqi_high - iaqi_low) / (bp_high - bp_low)
            * (concentration.loc[in_range] - bp_low) + iaqi_low
        )
    result.loc[concentration.eq(0)] = 0
    return result

daily_targets["CO_mg_m3"] = daily_targets["CO_24h_avg"] * 0.001
for pollutant in ["PM2.5", "PM10", "SO2", "NO2", "CO"]:
    concentration = daily_targets["CO_mg_m3"] if pollutant == "CO" else daily_targets[f"{pollutant}_24h_avg"]
    daily_targets[f"{pollutant}_iaqi"] = interpolate_iaqi(
        concentration, POLLUTANT_BREAKPOINTS[pollutant]
    ).round()
o3_iaqi = interpolate_iaqi(daily_targets["O3_8h_avg"], POLLUTANT_BREAKPOINTS["O3"])
o3_1h_iaqi = interpolate_iaqi(daily_targets["O3_1h_max"], O3_1H_BREAKPOINTS)
use_o3_1h = daily_targets["O3_8h_avg"].gt(800)
o3_iaqi.loc[use_o3_1h] = o3_1h_iaqi.loc[use_o3_1h]
daily_targets["O3_iaqi"] = o3_iaqi.round()

def category_from_iaqi(values):
    rounded = values.round()
    result = pd.Series(pd.NA, index=values.index, dtype="object")
    ranges = [(0, 50), (51, 100), (101, 150), (151, 200), (201, 300), (301, 500)]
    for code, (low, high) in enumerate(ranges, 1):
        result.loc[rounded.between(low, high)] = CATEGORY_NAMES[code]
    return result

iaqi_columns = [f"{pollutant}_iaqi" for pollutant in POLLUTANTS]
for pollutant in POLLUTANTS:
    daily_targets[f"{pollutant}_category"] = category_from_iaqi(daily_targets[f"{pollutant}_iaqi"])
daily_targets["aqi"] = daily_targets[iaqi_columns].max(axis=1, skipna=True)
daily_targets.loc[daily_targets[iaqi_columns].isna().all(axis=1), "aqi"] = np.nan
daily_targets["aqi_category"] = category_from_iaqi(daily_targets["aqi"])
daily_labels = daily_targets.reset_index()
TARGET_COLUMNS = ["aqi_category", *[f"{pollutant}_category" for pollutant in POLLUTANTS]]
print(f"Daily station records: {len(daily_labels):,}")
display(daily_labels[TARGET_COLUMNS].notna().sum().rename("valid_station_days").to_frame())


## 3. Recreate the exact hourly training and test sets

This repeats the station-based 80/20 split used in `sets.ipynb` with `RANDOM_STATE = 42`. It checks the requested source, training, and test row counts. The station-day target labels are then joined to every hourly row using station and date. Original hourly split sizes are printed before rows with missing labels are excluded from individual model fits.


In [ ]:
# Reproduce sets.ipynb: randomly select 20% of each station's rows for testing.
train_parts = []
test_parts = []
for _, group in source_df.groupby("station", dropna=False, sort=False):
    test_count = min(len(group) - 1, max(1, round(len(group) * TEST_FRACTION)))
    test_part = group.sample(n=test_count, random_state=RANDOM_STATE)
    train_parts.append(group.drop(index=test_part.index))
    test_parts.append(test_part)

train_raw = pd.concat(train_parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
test_raw = pd.concat(test_parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

actual_counts = (len(source_df), len(train_raw), len(test_raw))
expected_counts = (EXPECTED_SOURCE_ROWS, EXPECTED_TRAIN_ROWS, EXPECTED_TEST_ROWS)
print(f"Complete source rows: {len(source_df):,}")
print(f"Training set: {len(train_raw):,} rows")
print(f"Test set: {len(test_raw):,} rows")
print(f"Total after split: {len(train_raw) + len(test_raw):,} rows")
if actual_counts != expected_counts:
    raise ValueError(
        f"Split counts are {actual_counts}; expected {expected_counts}. "
        "Check that this is the same all_regions source and split logic as sets.ipynb."
    )

# Merge each station-day target back onto that station's hourly rows.
daily_label_columns = [*station_day_keys, *TARGET_COLUMNS]
def attach_daily_targets(rows):
    result = rows.copy()
    row_time = result[["year", "month", "day", "hour"]].apply(pd.to_numeric, errors="coerce")
    row_timestamp = pd.to_datetime({
        "year": row_time["year"], "month": row_time["month"],
        "day": row_time["day"], "hour": row_time["hour"],
    }, errors="coerce")
    result["_date"] = row_timestamp.dt.normalize()
    result["_station_key"] = result["station"].astype("string").str.strip().str.casefold()
    result = result.merge(
        daily_labels[daily_label_columns],
        on=station_day_keys,
        how="left",
        sort=False,
        validate="many_to_one",
    )
    result["_timestamp"] = row_timestamp
    return result

train_set = attach_daily_targets(train_raw)
test_set = attach_daily_targets(test_raw)
if (len(train_set), len(test_set)) != (EXPECTED_TRAIN_ROWS, EXPECTED_TEST_ROWS):
    raise RuntimeError("Attaching daily labels changed the split row counts unexpectedly.")

print("Hourly model input rows (before excluding rows without a valid target):")
print(f"Training partition: {len(train_set):,}")
print(f"Test partition: {len(test_set):,}")
print("Unique daily label rows are looked up and broadcast to these hourly rows.")

# Define hourly predictors explicitly; pollutant values and all AQI/target fields stay out.
WEATHER_FEATURES = [column for column in ["TEMP", "PRES", "DEWP", "RAIN", "WSPM"] if column in source_df.columns]
if not WEATHER_FEATURES:
    raise KeyError("No expected weather predictors were found (TEMP, PRES, DEWP, RAIN, WSPM).")
for frame in (train_set, test_set):
    frame["year"] = pd.to_numeric(frame["year"], errors="coerce")
    frame["month"] = pd.to_numeric(frame["month"], errors="coerce")
    frame["hour"] = pd.to_numeric(frame["hour"], errors="coerce")
    day_of_year = frame["_timestamp"].dt.dayofyear
    frame["month_sin"] = np.sin(2 * np.pi * (frame["month"] - 1) / 12)
    frame["month_cos"] = np.cos(2 * np.pi * (frame["month"] - 1) / 12)
    frame["day_of_year_sin"] = np.sin(2 * np.pi * (day_of_year - 1) / 365.25)
    frame["day_of_year_cos"] = np.cos(2 * np.pi * (day_of_year - 1) / 365.25)
    frame["hour_sin"] = np.sin(2 * np.pi * frame["hour"] / 24)
    frame["hour_cos"] = np.cos(2 * np.pi * frame["hour"] / 24)

NUMERIC_FEATURES = [
    *WEATHER_FEATURES, "year", "month_sin", "month_cos",
    "day_of_year_sin", "day_of_year_cos", "hour_sin", "hour_cos",
]
CATEGORICAL_FEATURES = ["station"] + (["wd"] if "wd" in source_df.columns else [])
FEATURE_COLUMNS = [*NUMERIC_FEATURES, *CATEGORICAL_FEATURES]
print("Predictor columns:", FEATURE_COLUMNS)
print("Target columns:", TARGET_COLUMNS)
print("Training rows with valid targets:")
display(train_set[TARGET_COLUMNS].notna().sum().rename("rows").to_frame())
print("Test rows with valid targets:")
display(test_set[TARGET_COLUMNS].notna().sum().rename("rows").to_frame())



## 4. Train and evaluate the Random Forest classifiers

One classifier is fitted for the overall AQI category and one for each pollutant category. The models use the exact hourly training partition above. For each target, rows without that target label are removed from that model’s fit/evaluation only; the original split remains unchanged. Preprocessing is fit on training rows only.


In [ ]:
models = {}
evaluation_rows = []

for target in TARGET_COLUMNS:
    train_task = train_set.dropna(subset=[target]).copy()
    test_task = test_set.dropna(subset=[target]).copy()
    if train_task.empty or test_task.empty:
        print(f"Skipping {target}: no labeled rows in train or test partition.")
        continue
    y_train = train_task[target].astype(str)
    y_test = test_task[target].astype(str)
    if y_train.nunique() < 2:
        print(f"Skipping {target}: training partition has fewer than two classes.")
        continue
    X_train = train_task[FEATURE_COLUMNS].copy()
    X_test = test_task[FEATURE_COLUMNS].copy()

    numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessing = ColumnTransformer([
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ])
    model = Pipeline([
        ("preprocessing", preprocessing),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    models[target] = model

    labels = sorted(set(y_train) | set(y_test))
    print(f"\n=== {target} ===")
    print(f"Train rows: {len(train_task):,} | Test rows: {len(test_task):,}")
    print(f"Accuracy: {accuracy_score(y_test, predictions):.3f}")
    print(f"Balanced accuracy: {balanced_accuracy_score(y_test, predictions):.3f}")
    display(pd.DataFrame(
        confusion_matrix(y_test, predictions, labels=labels),
        index=pd.Index(labels, name="actual"),
        columns=pd.Index(labels, name="predicted"),
    ))
    display(pd.DataFrame(classification_report(
        y_test, predictions, labels=labels, output_dict=True, zero_division=0
    )).T)
    feature_names = model.named_steps["preprocessing"].get_feature_names_out()
    importances = model.named_steps["classifier"].feature_importances_
    display(pd.Series(importances, index=feature_names).sort_values(ascending=False).head(10).rename("importance").to_frame())
    evaluation_rows.append({
        "target": target,
        "train_rows": len(train_task),
        "test_rows": len(test_task),
        "accuracy": accuracy_score(y_test, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
    })

model_results = pd.DataFrame(evaluation_rows).set_index("target")
display(model_results)
